# Занятие 28. Решение практики: ML-пайплайн на kNN

**Только для преподавателя.** Эталон к workflow_practice.ipynb.

Студентам выдавайте практику без этого файла.

---
## Дано: датасет

(та же ячейка, что у студентов)

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
X_all = data.data
y_all = data.target

print('Объектов:', X_all.shape[0], '| признаков:', X_all.shape[1])
print('Доля класса 1:', round(y_all.mean(), 3))
X_all.iloc[:, :5].head()

Объектов: 569 | признаков: 30
Доля класса 1: 0.627


,mean radius,mean texture,mean perimeter,mean area,mean smoothness
0,17.99,10.38,122.80,1001.0,0.11840
1,20.57,17.77,132.90,1326.0,0.08474
2,19.69,21.25,130.00,1203.0,0.10960
3,11.42,20.38,77.58,386.1,0.14250
4,20.29,14.34,135.10,1297.0,0.10030


---
## Задание 0. Постановка

In [2]:
task_spec = {
    'объект': 'опухоль',
    'цель': 'класс 0 (злокачественная) или 1 (доброкачественная)',
    'тип_задачи': 'классификация',
    'действие': 'направить на дообследование при подозрении на злокачественную',
}
PRIMARY_METRIC = 'recall'
task_spec, PRIMARY_METRIC

({'объект': 'опухоль',
  'цель': 'класс 0 (злокачественная) или 1 (доброкачественная)',
  'тип_задачи': 'классификация',
  'действие': 'направить на дообследование при подозрении на злокачественную'},
 'recall')

---
## Задание 1. Импорты и константы

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix

RANDOM_STATE = 42
TEST_SIZE = 0.2      # 20% test
VAL_SIZE = 0.25      # 25% от оставшегося -> 20% validation, 60% train

---
## Задание 2. Разбиение

In [4]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_all,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=VAL_SIZE, random_state=RANDOM_STATE, stratify=y_train_val,
)
print('train / val / test:', len(X_train), len(X_val), len(X_test))

train / val / test: 341 114 114


---
## Задание 3. EDA train

In [5]:
print('Доля класса 1 в train:', round(y_train.mean(), 3))
display(X_train[['mean radius', 'mean texture', 'mean area', 'mean smoothness']].describe())
print('std mean area:', round(X_train['mean area'].std(), 1))
print('std mean smoothness:', round(X_train['mean smoothness'].std(), 4))

Доля класса 1 в train: 0.628


,mean radius,mean texture,mean area,mean smoothness
count,341.000000,341.000000,341.000000,341.000000
mean,14.030343,19.183754,646.441935,0.096563
std,3.550771,4.528923,352.949179,0.013524
min,6.981000,9.710000,143.500000,0.062510
25%,11.600000,15.860000,412.600000,0.086750
50%,13.170000,18.750000,536.900000,0.096760
75%,15.490000,21.680000,744.900000,0.105400
max,28.110000,39.280000,2499.000000,0.142500


std mean area: 352.9
std mean smoothness: 0.0135


---
## Задание 4. Baseline

In [6]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
dummy_acc = accuracy_score(y_val, dummy.predict(X_val))
majority = max(y_train.mean(), 1 - y_train.mean())
print('dummy_acc:', round(dummy_acc, 3))
print('доля большего класса в train:', round(majority, 3))

dummy_acc: 0.623
доля большего класса в train: 0.628


---
## Задание 5. kNN без масштабирования

In [7]:
knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_raw.fit(X_train, y_train)
acc_raw = accuracy_score(y_val, knn_raw.predict(X_val))
print('acc_raw:', round(acc_raw, 3))
print('выше baseline?', acc_raw > dummy_acc)

acc_raw: 0.93
выше baseline? True


---
## Задание 6. kNN с масштабированием

In [8]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_s, y_train)
acc_scaled = accuracy_score(y_val, knn_scaled.predict(X_val_s))
print('acc_raw:', round(acc_raw, 3))
print('acc_scaled:', round(acc_scaled, 3))
print('прирост:', round(acc_scaled - acc_raw, 3))

acc_raw: 0.93
acc_scaled: 0.956
прирост: 0.026


---
## Задание 7. Подбор k

In [9]:
k_values = [1, 3, 5, 7, 11, 15, 21]
rows = []
for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_s, y_train)
    acc = accuracy_score(y_val, model.predict(X_val_s))
    rows.append({'k': k, 'val_acc': round(acc, 3)})

k_results = pd.DataFrame(rows)
best_k = int(k_results.loc[k_results['val_acc'].idxmax(), 'k'])
k_results, best_k

(    k  val_acc
 0   1    0.939
 1   3    0.965
 2   5    0.956
 3   7    0.956
 4  11    0.939
 5  15    0.939
 6  21    0.930,
 3)

---
## Задание 8. Анализ ошибок

In [10]:
best_knn = KNeighborsClassifier(n_neighbors=best_k)
best_knn.fit(X_train_s, y_train)
y_pred = best_knn.predict(X_val_s)

cm = confusion_matrix(y_val, y_pred)
print('confusion_matrix (строки=истина, столбцы=прогноз):')
print(cm)

# Положительный класс 0 = злокачественная
tp, fn, fp, tn = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
recall0 = recall_score(y_val, y_pred, pos_label=0)
print('TP FN FP TN (класс 0 положительный):', tp, fn, fp, tn)
print('recall класса 0:', round(recall0, 3))

median_radius = X_train['mean radius'].median()
groups = np.where(X_val['mean radius'] >= median_radius, 'крупные', 'мелкие')
errors = (y_pred != y_val.to_numpy())
error_by_group = pd.DataFrame({'группа': groups, 'ошибка': errors.astype(int)})
print(error_by_group.groupby('группа', observed=True)['ошибка'].mean().round(3))

confusion_matrix (строки=истина, столбцы=прогноз):
[[39  4]
 [ 0 71]]
TP FN FP TN (класс 0 положительный): 39 4 0 71
recall класса 0: 0.907
группа
крупные    0.048
мелкие     0.019
Name: ошибка, dtype: float64


---
## Задание 9. Test и новый объект

In [11]:
final_knn = KNeighborsClassifier(n_neighbors=best_k)
final_knn.fit(X_train_s, y_train)
test_acc = accuracy_score(y_test, final_knn.predict(X_test_s))
print('test accuracy:', round(test_acc, 3))

x_new = scaler.transform(X_test.iloc[[0]])
pred_new = final_knn.predict(x_new)[0]
true_new = y_test.iloc[0]
print('новый объект: прогноз', pred_new, '| истина', true_new)

test accuracy: 0.974


новый объект: прогноз 0 | истина 0
